In [1]:
import poolparty as pp
from poolparty.operations.fixed import fixed_operation, FixedOp

In [2]:
# Example 1: Direct use of fixed_operation() - custom uppercase transformation
with pp.Party() as party:
    parent = pp.from_seqs(['acgt', 'tgca'], mode='sequential')
    upper = fixed_operation(
        parents=[parent],
        seq_from_seqs_fn=lambda seqs: seqs[0].upper(),
        seq_length_from_pools_fn=lambda pools: pools[0].seq_length,
        name='upper'
    )
    df = upper.generate_library(num_cycles=1)
df

,seq,upper.seq,pool[0].seq,upper.state,pool[0].state,op[1]:fixed.state,op[0]:from_seqs.state,op[0]:from_seqs.key.seq_name,op[0]:from_seqs.key.seq_index
0,ACGT,ACGT,acgt,0,0,0,0,seq_0,0
1,TGCA,TGCA,tgca,1,1,0,1,seq_1,1


In [3]:
# Example 2: Custom concatenation with prefix/suffix
with pp.Party() as party:
    core = pp.from_seqs(['AAA', 'TTT'], mode='sequential')
    wrapped = fixed_operation(
        parents=[core],
        seq_from_seqs_fn=lambda seqs: f"[{seqs[0]}]",
        seq_length_from_pools_fn=lambda pools: pools[0].seq_length + 2 if pools[0].seq_length else None,
        name='wrapped'
    )
    df = wrapped.generate_library(num_cycles=1)
df

,seq,wrapped.seq,pool[0].seq,wrapped.state,pool[0].state,op[1]:fixed.state,op[0]:from_seqs.state,op[0]:from_seqs.key.seq_name,op[0]:from_seqs.key.seq_index
0,[AAA],[AAA],AAA,0,0,0,0,seq_0,0
1,[TTT],[TTT],TTT,1,1,0,1,seq_1,1


In [4]:
# Example 3: Multiple parents - interleave sequences character by character
with pp.Party() as party:
    a = pp.from_seq('AAAA')
    b = pp.from_seq('TTTT')
    interleaved = fixed_operation(
        parents=[a, b],
        seq_from_seqs_fn=lambda seqs: ''.join(x + y for x, y in zip(seqs[0], seqs[1])),
        seq_length_from_pools_fn=lambda pools: (pools[0].seq_length + pools[1].seq_length 
                                                  if pools[0].seq_length and pools[1].seq_length else None),
        name='interleaved'
    )
    df = interleaved.generate_library(num_seqs=1)
df

,seq,interleaved.seq,pool[1].seq,pool[0].seq,interleaved.state,pool[1].state,pool[0].state,op[2]:fixed.state,op[1]:fixed.state,op[0]:fixed.state
0,ATATATAT,ATATATAT,TTTT,AAAA,0,0,0,0,0,0


In [5]:
# Example 4: Built-in operations now use FixedOp internally
with pp.Party() as party:
    seq = pp.from_seq('ACGT')
    joined = pp.join([seq, '---', seq], name='joined')
    rc = pp.reverse_complement(seq, name='rc')
    swapped = pp.swapcase(seq, name='swapped')
    
    # All now use FixedOp
    print(f"from_seq uses: {type(seq.operation).__name__}")
    print(f"join uses: {type(joined.operation).__name__}")
    print(f"reverse_complement uses: {type(rc.operation).__name__}")
    print(f"swapcase uses: {type(swapped.operation).__name__}")

from_seq uses: FixedOp
join uses: FixedOp
reverse_complement uses: FixedOp
swapcase uses: FixedOp


In [6]:
# Example 5: Verify the refactored operations produce correct output
with pp.Party() as party:
    seq = pp.from_seq('AcGt')
    joined = pp.join(['[', seq, ']'], name='joined')
    rc = pp.reverse_complement(seq, name='rc')
    swapped = pp.swapcase(seq, name='swapped')

joined_df = joined.generate_library(num_seqs=1)
rc_df = rc.generate_library(num_seqs=1)
swapped_df = swapped.generate_library(num_seqs=1)

print(f"Original: AcGt")
print(f"Joined:   {joined_df['seq'].iloc[0]}")
print(f"Rev-comp: {rc_df['seq'].iloc[0]}")
print(f"Swapped:  {swapped_df['seq'].iloc[0]}")

Original: AcGt
Joined:   [AcGt]
Rev-comp: aCgT
Swapped:  aCgT
